In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
import pickle
import bz2
from glob import glob

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys, Descriptors, PandasTools, Draw
from rdkit.Chem.Draw import IPythonConsole
from rdkit.DataStructs import ExplicitBitVect

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, f1_score,
    roc_auc_score, cohen_kappa_score
)
from sklearn.model_selection import (
    train_test_split, RepeatedStratifiedKFold,
    ShuffleSplit, StratifiedShuffleSplit
)

from standardiser import break_bonds, neutralise, rules, unsalt
from standardiser.utils import StandardiseException, sanity_check

# Optional: untuk autoreload jika di Jupyter
# %reload_ext autoreload
# %autoreload 2

# Suppress warnings
warnings.filterwarnings("ignore")
warnings.warn = lambda *args, **kwargs: None

In [ ]:
import pandas as pd

# Fungsi untuk ubah string ke list of int
def string_to_list(bit_string):
    if isinstance(bit_string, str):
        return list(map(int, bit_string.strip('[]').split(', ')))
    return bit_string

# Load test set dari Excel
test_file = r"......"
test_df = pd.read_excel(test_file)

# Konversi kolom deskriptor jika masih berupa string
for col in ['Morgan_Descriptors', 'MACCS_Descriptors', 'APF_Descriptors']:
    if col in test_df.columns:
        if isinstance(test_df[col].iloc[0], str):
            test_df[col] = test_df[col].apply(string_to_list)

# Tampilkan hasil
print("Test DataFrame:")
print(test_df.head())


In [ ]:
# Melihat nama-nama kolom yang ada di DataFrame
print("Daftar kolom dalam test_df:")
print(test_df.columns.tolist())

In [ ]:
# Cek jumlah NaN sebelum dihapus
nan_before = test_df.isnull().sum().sum()

# Hapus baris yang mengandung NaN
test_df = test_df.dropna()

# Tampilkan informasi jumlah NaN
if nan_before > 0:
    print(f"Total nilai NaN yang dihapus dari test_df: {nan_before}")
else:
    print("Tidak ada nilai NaN yang ditemukan dalam test_df.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

# Buat salinan kolom Outcome
S = test_df['Outcome'].copy()

# Plot distribusi kelas
fig, ax = plt.subplots()
ax = S.hist(bins=np.arange(-0.5, 5), edgecolor='black')
ax.set_xticks(range(0, 5))
ax.set_xlabel("Outcome Class")
ax.set_ylabel("Count")
ax.set_title("Distribusi Outcome (Test Set)")
plt.show()

# Encoding label
le = LabelEncoder()
outcomes = np.unique(test_df['Outcome'])
le.fit(outcomes)
y = le.transform(test_df['Outcome'])

# Info distribusi
print("Classes                          :", outcomes)
print("Number of cpds in each class     :", np.bincount(y))
print("Total number of cpds             :", len(y))

# Ganti label Outcome menjadi angka (mapping)
S = test_df['Outcome']
info = {}
for i, cls in enumerate(S.unique()):
    info[cls] = i
    S = S.replace(cls, i)

# Optional: simpan mapping info kalau mau pakai nanti
print("Label mapping (kelas → angka):", info)

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Ambil label Outcome dari test_df
S = test_df['Outcome'].copy()

# Encode label ke angka
info = {}
for i, cls in enumerate(S.unique()):
    info[cls] = i
    S = S.replace(cls, i)

# Konversi label ke numpy array bertipe int32
y_test = np.int32(S)

# Konversi MACCS, Morgan, dan APF Descriptors ke array numpy
def convert_to_array(desc_list):
    return np.array([eval(desc) if isinstance(desc, str) else desc for desc in desc_list])

x_test_macckeys = convert_to_array(test_df['MACCS_Descriptors'])
x_test_morgan = convert_to_array(test_df['Morgan_Descriptors'])
x_test_apf = convert_to_array(test_df['APF_Descriptors'])  # <-- tambahan APF

# Cek isi
print("Label classes (encoded)       :", info)
print("Jumlah senyawa per kelas      :", np.bincount(y_test))
print("Total jumlah senyawa (test)   :", len(y_test))
print("x_test_macckeys shape         :", x_test_macckeys.shape)
print("x_test_morgan shape           :", x_test_morgan.shape)
print("x_test_apf shape              :", x_test_apf.shape)  # <-- cek APF


In [ ]:
x_rdkitcdk = test_df.drop(columns=['SMILES',
    'Outcome',
    'Morgan_Descriptors',
    'MACCS_Descriptors',
    'APF_Descriptors'])
x_rdkitcdk

In [ ]:
print(x_rdkitcdk)

In [ ]:
x_rdkitcdk  = x_rdkitcdk.apply(lambda row: row.values, axis=1).tolist()

# Add the new column 'rdkit_cdk' to test_df
test_df['rdkit_cdk'] = x_rdkitcdk 

# Display the updated DataFrame
print(test_df)

In [ ]:
y_test = np.int32(S)
x_test_morgan = np.array(list(test_df['Morgan_Descriptors']))
x_test_macckeys = np.array(list(test_df['MACCS_Descriptors']))
x_test_rdkit_cdk = np.array(list(test_df['rdkit_cdk']))
x_test_apf = np.array(list(test_df['APF_Descriptors']))  # <-- tambahan APF


In [ ]:
y_test= np.int32((S))
x_test_rdkit_cdk

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, f1_score, classification_report


In [ ]:
y_true = test_df['Outcome'].astype(int)  # Ensure it's of integer type, suitable for metrics calculation


In [ ]:
test_df

In [ ]:
def convert_list_str_to_float(lst):
    return [float(x) for x in lst if x != '' and x is not None]

test_df['rdkit_cdk'] = test_df['rdkit_cdk'].apply(convert_list_str_to_float)

X_rdkitcdk = np.array(test_df['rdkit_cdk'].tolist(), dtype=float)

# Evaluation 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score
import joblib
import itertools
import os
import ast
from tensorflow.keras.models import load_model

# ============================================================
# 0. PATHS & GLOBAL CONFIGURATION
# ============================================================
# Path to the independent / external test set
test_files = r"......"  # TODO: replace with your file path

# Path to the file containing training 10-fold CV AUC for each model
# The file must contain at least the columns: ['Fingerprint', 'Algorithm', 'AUC_CV']
auc_cv_path = r"......\training_cv_auc.xlsx"  # TODO: replace with your file path

# Endpoint name (used in file naming only)
endpoint = "Hepatotoxicity"  # e.g., "Hepatotoxicity", "AcuteDermal", etc.

# Model paths: update {endpoint} and root paths according to your directory structure
models_info = {
    'SVM': {
        'Morgan':   fr"path\to\{endpoint}_SVM_Morgan.pkl",
        'MACCS':    fr"path\to\{endpoint}_SVM_MACCS.pkl",
        'APF':      fr"path\to\{endpoint}_SVM_APF.pkl",
        'RDKitCDK': fr"path\to\{endpoint}_SVM_RDKitCDK.pkl"
    },
    'RF': {
        'Morgan':   fr"path\to\{endpoint}_RF_Morgan.pkl",
        'MACCS':    fr"path\to\{endpoint}_RF_MACCS.pkl",
        'APF':      fr"path\to\{endpoint}_RF_APF.pkl",
        'RDKitCDK': fr"path\to\{endpoint}_RF_RDKitCDK.pkl"
    },
    'XGB': {
        'Morgan':   fr"path\to\{endpoint}_XGB_Morgan.pkl",
        'MACCS':    fr"path\to\{endpoint}_XGB_MACCS.pkl",
        'APF':      fr"path\to\{endpoint}_XGB_APF.pkl",
        'RDKitCDK': fr"path\to\{endpoint}_XGB_RDKitCDK.pkl"
    },
    'DNN': {
        'Morgan':   fr"path\to\{endpoint}_DNN_Morgan.pth",
        'MACCS':    fr"path\to\{endpoint}_DNN_MACCS.pth",
        'APF':      fr"path\to\{endpoint}_DNN_APF.pth",
        'RDKitCDK': fr"path\to\{endpoint}_DNN_RDKitCDK.pth"
    }
}

# Threshold for selecting top models in the multi‑algorithm consensus
threshold_auc = 0.80  # can be adjusted (e.g., different threshold or top‑N strategy)

# Output path for the final evaluation results
save_path = rf"C:\UserPC\{endpoint}\Evaluation\Evaluation_Individual_SingleAlgoConsensus_MultiAlgoConsensus.xlsx"

# ============================================================
# 1. LOAD TEST SET AND PREPARE DESCRIPTORS
# ============================================================
test_df = pd.read_excel(test_files)

# Columns not used as RDKit/CDK descriptors
drop_cols = ['SMILES', 'Morgan_Descriptors', 'MACCS_Descriptors',
             'APF_Descriptors', 'Outcome']

# RDKit/CDK descriptor matrix (physicochemical descriptors)
x_rdkitcdk_test = test_df.drop(columns=drop_cols)

# True binary labels (0 = non‑toxic, 1 = toxic)
y_true = test_df['Outcome'].astype(int).values

# Fingerprint and algorithm lists
fingerprints = ['Morgan', 'MACCS', 'APF', 'RDKitCDK']
algorithms = ['SVM', 'RF', 'XGB', 'DNN']

def convert_to_array(series):
    """
    Convert a pandas Series of stringified lists into a 2D NumPy array.

    Example input cell: "[0, 1, 0, 1, ...]"
    """
    return np.array(series.apply(ast.literal_eval).tolist())

# Precompute descriptor arrays to avoid repeated conversions
X_morgan   = convert_to_array(test_df["Morgan_Descriptors"])
X_maccs    = convert_to_array(test_df["MACCS_Descriptors"])
X_apf      = convert_to_array(test_df["APF_Descriptors"])
X_rdkitcdk = x_rdkitcdk_test.values

X_dict = {
    'Morgan':   X_morgan,
    'MACCS':    X_maccs,
    'APF':      X_apf,
    'RDKitCDK': X_rdkitcdk
}

# ============================================================
# 2. LOAD ALL TRAINED MODELS
# ============================================================
loaded_models = {}

for algo, fps in models_info.items():
    loaded_models[algo] = {}
    for fp_name, path in fps.items():
        if algo == 'DNN':
            # Deep learning models saved in Keras format (.keras)
            loaded_models[algo][fp_name] = load_model(path)  # [web:22][web:25][web:28]
        else:
            # Classical ML models saved with joblib (.pkl)
            loaded_models[algo][fp_name] = joblib.load(path)

print("All models were successfully loaded.\n")

# ============================================================
# 3. LOAD TRAINING CV AUC VALUES AND BUILD WEIGHT DICTIONARY
# ============================================================
auc_df = pd.read_excel(auc_cv_path)

# Expected columns: ['Fingerprint', 'Algorithm', 'AUC_CV']
auc_weights = {}
for _, row in auc_df.iterrows():
    key = (row['Fingerprint'], row['Algorithm'])
    auc_weights[key] = float(row['AUC_CV'])

# ============================================================
# 4. UTILITY FUNCTIONS
# ============================================================
def get_probs(model, algo, fp, X_dict):
    """
    Obtain the predicted probability of the positive (toxic) class
    for a given model, algorithm type, and fingerprint representation.

    For DNN models, model.predict is assumed to output probabilities.
    For classical ML models (SVM/RF/XGB), predict_proba is used. [web:25][web:28]
    """
    X_input = X_dict[fp]
    if algo == 'DNN':
        probs = model.predict(X_input).ravel()
    else:
        probs = model.predict_proba(X_input)[:, 1]
    return probs

def bootstrap_metrics(probs, y_true, n_bootstrap=1000, ci=0.95, random_state=42):
    """
    Estimate classification metrics and their 95% confidence intervals
    using bootstrap resampling (with replacement).

    Metrics: AUC, overall accuracy (ACC), sensitivity (SEN),
    specificity (SPE), and mean counts of TN, FP, FN, TP. [web:1][web:2][web:21][web:24]
    """
    rng = np.random.RandomState(random_state)
    preds = (probs >= 0.5).astype(int)

    acc_list, sen_list, spe_list, auc_list = [], [], [], []
    tn_list, fp_list, fn_list, tp_list = [], [], [], []
    n_samples = len(y_true)

    for _ in range(n_bootstrap):
        # Sample indices with replacement
        idx = rng.choice(np.arange(n_samples), size=n_samples, replace=True)
        y_sample = y_true[idx]
        p_sample = probs[idx]
        pred_sample = preds[idx]

        # Force a 2x2 confusion matrix structure via labels=[0,1] [web:21][web:24][web:30]
        tn, fp, fn, tp = confusion_matrix(y_sample, pred_sample, labels=[0, 1]).ravel()
        tn_list.append(tn)
        fp_list.append(fp)
        fn_list.append(fn)
        tp_list.append(tp)

        # Accuracy
        acc_list.append(accuracy_score(y_sample, pred_sample))
        # Sensitivity (TPR)
        sen_list.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        # Specificity (TNR)
        spe_list.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

        # AUC may fail if only a single class is present in the bootstrap sample
        try:
            auc_list.append(roc_auc_score(y_sample, p_sample))
        except:
            auc_list.append(np.nan)

    lower = (1 - ci) / 2
    upper = 1 - lower

    def format_metric(values):
        """
        Return 'mean ± half_width', where half_width corresponds
        to half of the 95% bootstrap percentile interval width. [web:2][web:23]
        """
        mean_val = np.nanmean(values)
        low = np.nanpercentile(values, 100 * lower)
        high = np.nanpercentile(values, 100 * upper)
        half_width = (high - low) / 2
        return f"{mean_val:.2f} ± {half_width:.2f}"

    metrics = {
        'AUC':         format_metric(auc_list),
        'Accuracy':    format_metric(acc_list),
        'Sensitivity': format_metric(sen_list),
        'Specificity': format_metric(spe_list),
        'TN': int(np.mean(tn_list)),
        'FP': int(np.mean(fp_list)),
        'FN': int(np.mean(fn_list)),
        'TP': int(np.mean(tp_list))
    }
    return metrics

# ============================================================
# 5. MODEL EVALUATION:
#    (i) INDIVIDUAL
#    (ii) SINGLE‑ALGORITHM CONSENSUS (WEIGHTED)
#    (iii) MULTI‑ALGORITHM CONSENSUS (WEIGHTED)
# ============================================================
results_list = []

# ------------------------------------------------------------
# (i) Individual modeling: single fingerprint + single algorithm
# ------------------------------------------------------------
for algo in algorithms:
    for fp in fingerprints:
        model = loaded_models[algo][fp]
        probs = get_probs(model, algo, fp, X_dict)
        metrics = bootstrap_metrics(probs, y_true)
        metrics['Type'] = "Individual"
        metrics['Combination'] = f"{fp}-{algo}"
        results_list.append(metrics)

# -----------------------------------------------------------------
# (ii) Single‑algorithm consensus:
#      For each algorithm, combine the four descriptor‑specific models
#      (Morgan, MACCS, APF, RDKitCDK) using AUC‑based weights.
#      w_j = AUC_j / sum_k AUC_k  over models in the same algorithm. [web:2][web:15]
# -----------------------------------------------------------------
for algo in algorithms:
    probs_list = []
    weight_list = []
    fp_used = []

    for fp in fingerprints:
        key = (fp, algo)
        if key not in auc_weights:
            # Skip models without available training AUC
            continue
        model = loaded_models[algo][fp]
        probs = get_probs(model, algo, fp, X_dict)
        w = auc_weights[key]
        probs_list.append(probs)
        weight_list.append(w)
        fp_used.append(fp)

    if len(probs_list) == 0:
        continue

    weight_arr = np.array(weight_list)
    weight_arr = weight_arr / weight_arr.sum()  # Normalize weights to sum to 1

    probs_stack = np.vstack(probs_list)  # Shape: (n_models, n_samples)
    consensus_probs = np.average(probs_stack, axis=0, weights=weight_arr)

    metrics = bootstrap_metrics(consensus_probs, y_true)
    metrics['Type'] = "SingleAlgorithmConsensus"
    metrics['Combination'] = f"SingleAlgoConsensus-{algo}_" + "+".join(fp_used)
    results_list.append(metrics)

# -----------------------------------------------------------------
# (iii) Multi‑algorithm (multi‑modality) consensus:
#       Combine multiple top‑performing models (across algorithms and
#       fingerprints) using AUC‑based weights, restricted to models
#       with training AUC_CV >= threshold_auc. [web:2][web:15]
# -----------------------------------------------------------------
top_models = [
    (fp, algo) for (fp, algo), aucv in auc_weights.items()
    if aucv >= threshold_auc
]

probs_list = []
weight_list = []
combo_used = []

for fp, algo in top_models:
    model = loaded_models[algo][fp]
    probs = get_probs(model, algo, fp, X_dict)
    w = auc_weights[(fp, algo)]
    probs_list.append(probs)
    weight_list.append(w)
    combo_used.append(f"{fp}-{algo}")

if len(probs_list) > 0:
    weight_arr = np.array(weight_list)
    weight_arr = weight_arr / weight_arr.sum()

    probs_stack = np.vstack(probs_list)
    consensus_probs = np.average(probs_stack, axis=0, weights=weight_arr)

    metrics = bootstrap_metrics(consensus_probs, y_true)
    metrics['Type'] = "MultiAlgorithmConsensus"
    metrics['Combination'] = "MultiAlgoConsensus_TopModels_" + "+".join(combo_used)
    results_list.append(metrics)

# ============================================================
# 6. SAVE RESULTS TO EXCEL (SORTED BY MEAN AUC)
# ============================================================
metrics_df = pd.DataFrame(results_list)

# Extract the mean AUC value (first number before '±') for sorting
metrics_df["AUC_val"] = metrics_df["AUC"].str.extract(r"^([0-9.]+)").astype(float)

metrics_df = metrics_df.sort_values(by="AUC_val", ascending=False).drop(columns=["AUC_val"])
metrics_df = metrics_df[['Type', 'Combination', 'AUC', 'Accuracy',
                         'Sensitivity', 'Specificity', 'TN', 'FP', 'FN', 'TP']]

os.makedirs(os.path.dirname(save_path), exist_ok=True)
metrics_df.to_excel(save_path, index=False)

print(f"Evaluation results for {len(metrics_df)} models "
      f"(Individual + Single‑Algorithm Consensus + Multi‑Algorithm Consensus) "
      f"were saved to:\n{save_path}")
